# PI4 — Ingestão e QA do Censo Escolar 2023–2025

Notebook operacional para reproduzir o painel escola-ano usado no projeto. Ele lê os arquivos oficiais preservados no Google Drive, filtra escolas ativas de São Paulo, harmoniza 2023–2025 e executa as checagens estruturais do pipeline.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys

PROJECT_ROOT = Path('/content/drive/MyDrive/UNIVESP — PI4 — Infraestrutura Escolar — Guaratinguetá')
DATA_ROOT = PROJECT_ROOT / '01_Dados'
SOURCE = DATA_ROOT / '1_fonte_original'
OUTPUT = DATA_ROOT / '2_tratamentos_dados' / 'base_analitica'
OUTPUT.mkdir(parents=True, exist_ok=True)

assert PROJECT_ROOT.exists(), f'Ajuste PROJECT_ROOT. Pasta não encontrada: {PROJECT_ROOT}'
print(PROJECT_ROOT)


In [ ]:
# Usa sempre o código versionado do repositório oficial.
!rm -rf /content/pi4_repo
!git clone -q --depth 1 https://github.com/felipecsr/univesp-projeto-integrador-4.git /content/pi4_repo
sys.path.insert(0, '/content/pi4_repo')

from src.censo_pipeline import build_panel, validate_panel, qa_summary, variable_qa, save_outputs


In [ ]:
zip_2023 = SOURCE / '2023' / 'microdados_censo_escolar_2023.zip'
zip_2024 = SOURCE / '2024' / 'microdados_censo_escolar_2024.zip'
escola_2025 = SOURCE / '2025' / 'Tabela_Escola_2025_V2.csv'
matricula_2025 = SOURCE / '2025' / 'Tabela_Matricula_2025_V2.csv'

for path in [zip_2023, zip_2024, escola_2025, matricula_2025]:
    assert path.exists(), f'Fonte não encontrada: {path}'

panel = build_panel(
    zip_2023,
    zip_2024,
    school_2025=escola_2025,
    matricula_2025=matricula_2025,
)
validate_panel(panel)
qa_summary(panel)


In [ ]:
# QA detalhado das variáveis do núcleo.
qa_vars = variable_qa(panel)
qa_vars


In [ ]:
# Grava artefatos reproduzíveis em 01_Dados/2_tratamentos_dados/base_analitica.
paths = save_outputs(panel, OUTPUT)
paths


In [ ]:
# Conferência rápida do município-foco.
guara = panel[panel['CO_MUNICIPIO'] == '3518404']
guara.groupby('NU_ANO_CENSO').agg(
    escolas=('CO_ENTIDADE', 'nunique'),
    matriculas=('QT_MAT_BAS', 'sum'),
).reset_index()
